In [4]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 2. Cargar datos con tipos forzados (Evita que el FIPS pierda el cero inicial)
path_votos = '/content/drive/MyDrive/Proyecto_Elecciones_EEUU/countypres_2000-2020.csv'
path_censo = '/content/drive/MyDrive/Proyecto_Elecciones_EEUU/acs2015_census_tract_data.csv'

df_votos = pd.read_csv(path_votos, dtype={'county_fips': str})
df_censo = pd.read_csv(path_censo, dtype={'CensusId': str})

In [10]:
# Filtramos usando el valor exacto que vimos en tu captura de pantalla
# Filtro con el valor real del archivo: 'US PRESIDENT'
df_2020 = df_votos[(df_votos['year'] == 2020) & (df_votos['office'] == 'US PRESIDENT')].copy()

# Normalizamos el FIPS de votos a 5 dígitos para que coincida con el censo
df_2020['county_fips'] = df_2020['county_fips'].astype(str).str.zfill(5)

# Optimización para GPU
df_2020['county_fips'] = df_2020['county_fips'].astype('category')
for col in ['Hispanic', 'White', 'Black', 'Native', 'Asian']:
    df_censo_county[col] = df_censo_county[col].astype('float32')

In [9]:
# 1. Aseguramos que el CensusTract tenga 11 dígitos (rellenando con el cero del estado si falta)
df_censo['CensusTract'] = df_censo['CensusTract'].astype(str).str.zfill(11)

# 2. El FIPS del Condado son los primeros 5 dígitos del CensusTract
df_censo['county_fips'] = df_censo['CensusTract'].str[:5]

# 3. Agregamos: Sumamos poblaciones y promediamos porcentajes demográficos
df_censo_county = df_censo.groupby('county_fips').agg({
    'TotalPop': 'sum',
    'Hispanic': 'mean',
    'White': 'mean',
    'Black': 'mean',
    'Native': 'mean',
    'Asian': 'mean'
}).reset_index()

print(df_censo_county)

     county_fips  TotalPop   Hispanic      White      Black    Native  \
0          01001     55221   2.850000  73.150000  20.825000  0.566667   
1          01003    195121   4.064516  83.454839   9.625806  0.600000   
2          01005     26932   4.800000  46.611111  46.488889  0.144444   
3          01007     22604   2.475000  77.500000  17.725000  0.450000   
4          01009     57710   8.888889  87.800000   1.355556  0.277778   
...          ...       ...        ...        ...        ...       ...   
3215       72145     56858  96.438462   3.300000   0.146154  0.000000   
3216       72147      9130  96.200000   3.400000   0.000000  0.000000   
3217       72149     24685  99.666667   0.016667   0.000000  0.000000   
3218       72151     36279  99.725000   0.212500   0.000000  0.000000   
3219       72153     39474  99.509091   0.436364   0.000000  0.000000   

         Asian  
0     0.700000  
1     0.658065  
2     0.333333  
3     0.150000  
4     0.122222  
...        ...  
3215

In [11]:
print(len(df_2020) )

22093


In [13]:
print(len(df_censo_county))

3220


In [24]:
# 1. Filtramos por el año y cargo correcto (visto en tu captura)
df_2020 = df_votos[(df_votos['year'] == 2020) & (df_votos['office'] == 'US PRESIDENT')].copy()

# 2. Normalizamos FIPS a 5 dígitos (arreglando el "10" que mencionaste)
df_2020['county_fips'] = df_2020['county_fips'].astype(str).str.zfill(5)

# 3. PIVOTE: Pasamos de formato largo a ancho
votos_pivote = df_2020.pivot_table(
    index='county_fips',
    columns='party',
    values='candidatevotes',
    aggfunc='sum',
    observed=False
).reset_index()

# 4. RECUPERACIÓN: Re-inyectamos 'totalvotes' (esto evita el KeyError)
df_totales = df_2020[['county_fips', 'totalvotes']].drop_duplicates()
df_votos_ready = pd.merge(votos_pivote, df_totales, on='county_fips', how='inner')

In [25]:
# Unión final
df_final = pd.merge(votos_pivote, df_censo_county, on='county_fips', how='inner')

# Guardar el progreso para mañana
df_final.to_csv('/content/drive/MyDrive/Proyecto_Elecciones_EEUU/dataset_final_limpio.csv', index=False)
print("¡Archivo maestro guardado y listo para el análisis!")

¡Archivo maestro guardado y listo para el análisis!


In [26]:
print(df_final)

     county_fips  DEMOCRAT  GREEN  LIBERTARIAN   OTHER  REPUBLICAN  TotalPop  \
0          01001    7503.0    NaN          NaN   429.0     19838.0     55221   
1          01003   24578.0    NaN          NaN  1557.0     83544.0    195121   
2          01005    4816.0    NaN          NaN    80.0      5622.0     26932   
3          01007    1986.0    NaN          NaN    84.0      7525.0     22604   
4          01009    2640.0    NaN          NaN   237.0     24711.0     57710   
...          ...       ...    ...          ...     ...         ...       ...   
3110       56037    3823.0    NaN        350.0   296.0     12229.0     44772   
3111       56039    9848.0    NaN        255.0   343.0      4341.0     22311   
3112       56041    1591.0    NaN        172.0   200.0      7496.0     20930   
3113       56043     651.0    NaN         65.0    71.0      3245.0      8400   
3114       56045     360.0    NaN         46.0    47.0      3107.0      7152   

       Hispanic      White      Black  

In [27]:
# Unión final
df_final = pd.merge(df_votos_ready, df_censo_county, on='county_fips', how='inner')

# --- VALIDACIÓN DE CALIDAD ---
# A. ¿Los votos cuadran?
df_final['votos_otros'] = df_final['totalvotes'] - (df_final['DEMOCRAT'] + df_final['REPUBLICAN'])

# B. ¿La demografía suma ~100%?
etnias = ['Hispanic', 'White', 'Black', 'Native', 'Asian']
df_final['check_etnias'] = df_final[etnias].sum(axis=1)

print(f"Dataset listo con {len(df_final)} condados.")
print(f"Suma promedio de etnias: {df_final['check_etnias'].mean():.2f}%")

Dataset listo con 3115 condados.
Suma promedio de etnias: 98.07%


In [29]:
# 1. Identificamos qué columnas son de partidos (las que están en mayúsculas por el pivote)
# Normalmente: 'DEMOCRAT', 'REPUBLICAN', 'LIBERTARIAN', etc.
columnas_partidos = df_final.columns[df_final.columns.isin(['DEMOCRAT', 'REPUBLICAN', 'LIBERTARIAN', 'GREEN', 'OTHER'])]

# 2. Cambiamos los NaN por 0 (Imputación)
df_final[columnas_partidos] = df_final[columnas_partidos].fillna(0)

# 3. Verificamos que ya no existan nulos en esas columnas
print("Nulos restantes en partidos:", df_final[columnas_partidos].isnull().sum().sum())

# Mostramos las primeras filas para confirmar visualmente
df_final[['county_fips', 'DEMOCRAT', 'REPUBLICAN', 'GREEN','totalvotes']].head()

Nulos restantes en partidos: 0


,county_fips,DEMOCRAT,REPUBLICAN,GREEN,totalvotes
0,01001,7503.0,19838.0,0.0,27770
1,01003,24578.0,83544.0,0.0,109679
2,01005,4816.0,5622.0,0.0,10518
3,01007,1986.0,7525.0,0.0,9595
4,01009,2640.0,24711.0,0.0,27588


In [30]:
# Esto te dirá qué partidos hay realmente en 2020 para Presidente
print(df_2020['party'].unique())

['DEMOCRAT' 'OTHER' 'REPUBLICAN' 'GREEN' 'LIBERTARIAN']


In [35]:
# Buscamos 'GREEN' en cualquier parte del texto de la columna party
check_sucio = df_votos[df_votos['party'].str.contains('GREEN', na=False, case=False)]
print(f"Registros encontrados con la palabra GREEN: {len(check_sucio)}")

Registros encontrados con la palabra GREEN: 6035


In [36]:
# 1. Recarga limpia (por si acaso)
df_votos = pd.read_csv(path_votos, low_memory=False)

# 2. LIMPIEZA CRÍTICA: Quitamos espacios en blanco en las columnas de texto
df_votos['party'] = df_votos['party'].str.strip()
df_votos['office'] = df_votos['office'].str.strip()
df_votos['state'] = df_votos['state'].str.strip()

# 3. Normalizamos el FIPS (Asegurando los 5 dígitos)
df_votos['county_fips'] = df_votos['county_fips'].astype(str).str.split('.').str[0].str.zfill(5)

In [37]:
# Pivote dinámico
votos_pivote = df_2020.pivot_table(
    index='county_fips',
    columns='party',
    values='candidatevotes',
    aggfunc='sum'
).reset_index().fillna(0)

# Recuperar totalvotes y unir con censo
df_totales = df_2020[['county_fips', 'totalvotes']].drop_duplicates()
df_votos_ready = pd.merge(votos_pivote, df_totales, on='county_fips', how='inner')
df_final = pd.merge(df_votos_ready, df_censo_county, on='county_fips', how='inner')

print(f"Dataset final consolidado. Condados: {len(df_final)}")

Dataset final consolidado. Condados: 3115


In [38]:
# 1. Definimos el orden lógico de las columnas
columnas_identidad = ['county_fips', 'totalvotes']
# Identificamos dinámicamente los partidos presentes
columnas_votos = [c for c in df_final.columns if c in ['DEMOCRAT', 'REPUBLICAN', 'GREEN', 'LIBERTARIAN', 'OTHER']]
columnas_censo = ['TotalPop', 'Hispanic', 'White', 'Black', 'Native', 'Asian']

# 2. Reordenamos el DataFrame
df_maestro = df_final[columnas_identidad + columnas_votos + columnas_censo].copy()

# 3. Formateo de nombres (opcional pero profesional: todo en minúsculas y sin espacios)
df_maestro.columns = [c.lower() for c in df_maestro.columns]

# 4. Guardado en Drive
path_salida = '/content/drive/MyDrive/Proyecto_Elecciones_EEUU/dataset_final_maestro.csv'
df_maestro.to_csv(path_salida, index=False)

print(f"✅ Archivo guardado con éxito en: {path_salida}")
print(f"Estructura final: {df_maestro.columns.tolist()}")

✅ Archivo guardado con éxito en: /content/drive/MyDrive/Proyecto_Elecciones_EEUU/dataset_final_maestro.csv
Estructura final: ['county_fips', 'totalvotes', 'democrat', 'green', 'libertarian', 'other', 'republican', 'totalpop', 'hispanic', 'white', 'black', 'native', 'asian']


In [39]:
import pandas as pd

# 1. Recarga absoluta: le decimos que todo es String para que no ignore nada
df_votos = pd.read_csv(path_votos, dtype=str)

# 2. Limpieza profunda: eliminamos espacios invisibles y pasamos a mayúsculas
df_votos['party'] = df_votos['party'].str.strip().str.upper()
df_votos['office'] = df_votos['office'].str.strip().str.upper()
df_votos['county_fips'] = df_votos['county_fips'].str.split('.').str[0].str.zfill(5)

# 3. Convertimos a número solo la columna de votos (manejando errores)
df_votos['candidatevotes'] = pd.to_numeric(df_votos['candidatevotes'], errors='coerce').fillna(0)

In [40]:
# Buscamos por patrón para no fallar
hallazgo_especifico = df_votos[
    (df_votos['county_fips'] == '01010') &
    (df_votos['party'].str.contains('GREEN', na=False))
]

print("Resultado de la auditoría en 01010:")
print(hallazgo_especifico[['year', 'office', 'party', 'candidatevotes']])

Resultado de la auditoría en 01010:
Empty DataFrame
Columns: [year, office, party, candidatevotes]
Index: []


In [41]:
# --- PASO A: CARGA LIMPIA DE VOTOS ---
df_votos = pd.read_csv(path_votos, low_memory=False)

# Limpiamos los FIPS: quitamos decimales, espacios y aseguramos 5 dígitos
df_votos['county_fips'] = df_votos['county_fips'].astype(str).str.split('.').str[0].str.zfill(5)
df_votos['party'] = df_votos['party'].str.strip().str.upper()

# Filtramos solo 2020 y Presidente
df_2020 = df_votos[(df_votos['year'] == '2020') & (df_votos['office'] == 'US PRESIDENT')].copy()

# --- PASO B: PIVOTE DINÁMICO (Para no perder a GREEN ni a nadie) ---
votos_pivote = df_2020.pivot_table(
    index='county_fips',
    columns='party',
    values='candidatevotes',
    aggfunc='sum'
).reset_index().fillna(0)

# --- PASO C: UNIÓN CON EL CENSO (Asegurando coincidencia de FIPS) ---
# El censo también debe tener FIPS de 5 dígitos
df_censo_county['county_fips'] = df_censo_county['county_fips'].astype(str).str.zfill(5)

df_final_maestro = pd.merge(votos_pivote, df_censo_county, on='county_fips', how='inner')

# --- PASO D: GUARDADO PROFESIONAL ---
df_final_maestro.to_csv('/content/drive/MyDrive/dataset_final_arreglado.csv', index=False)

print(f"¡Arreglado! Archivo guardado con {len(df_final_maestro)} condados.")

¡Arreglado! Archivo guardado con 0 condados.


In [42]:
# Busca por nombre para encontrar el FIPS real que tiene Python
busqueda_nombre = df_votos[df_votos['county_name'].str.contains('ETOWAH', na=False, case=False)]
print(busqueda_nombre[['county_name', 'county_fips', 'party', 'candidatevotes']].head())

      county_name county_fips       party  candidatevotes
108        ETOWAH       01055    DEMOCRAT           17433
109        ETOWAH       01055  REPUBLICAN           21087
110        ETOWAH       01055       GREEN             435
111        ETOWAH       01055       OTHER             393
12545      ETOWAH       01055    DEMOCRAT           15328
